In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import kurtosis, skew
import time

DATA_DIR = Path("/mnt/d/2èmeBD/XJTU-SY_Bearing_Datasets/40Hz10kN/Bearing3_1")
files    = sorted(DATA_DIR.glob("*.csv"), key=lambda f: int(f.stem))

SAMPLING_RATE = 25_600
WINDOW_SIZE   = 1024
WINDOW_STEP   = 512

# 1 fichier = 1 minute
WARMUP_END  = 30    # fichiers 1→30   = 30 min rodage
COLLECT_END = 300   # fichiers 31→120 = 1h30 collecte on est passé à 300 pour le teste
# détection  : fichiers 121→fin

print(f"Total fichiers     : {len(files)}")
print(f"Rodage             : fichiers 1 → {WARMUP_END}  ({WARMUP_END} min)")
print(f"Collecte           : fichiers {WARMUP_END+1} → {COLLECT_END}  ({COLLECT_END-WARMUP_END} min = 1h30)")
print(f"Détection          : fichiers {COLLECT_END+1} → {len(files)}  ({len(files)-COLLECT_END} min)")

In [ ]:
FEATURE_NAMES = [
    "mean", "std", "rms", "max", "min", "peak_to_peak",
    "skewness", "kurtosis", "crest_factor", "shape_factor",
    "impulse_factor", "freq_dominant", "spectral_energy", "spectral_entropy",
]

def extract_features(window, fs=SAMPLING_RATE):
    eps = 1e-10
    n = len(window)
    mean_val     = np.mean(window)
    std_val      = np.std(window)
    rms_val      = np.sqrt(np.mean(window**2))
    max_val      = np.max(np.abs(window))
    min_val      = np.min(np.abs(window))
    p2p_val      = np.ptp(window)
    skew_val     = float(skew(window))
    kurt_val     = float(kurtosis(window))
    mean_abs     = np.mean(np.abs(window))
    crest_val    = max_val / (rms_val + eps)
    shape_val    = rms_val / (mean_abs + eps)
    impulse_val  = max_val / (mean_abs + eps)
    fft_mag      = np.abs(np.fft.rfft(window))
    freqs        = np.fft.rfftfreq(n, d=1.0/fs)
    freq_dom     = freqs[np.argmax(fft_mag)]
    spec_energy  = np.sum(fft_mag**2)
    psd_norm     = fft_mag**2 / (spec_energy + eps)
    spec_entropy = -np.sum(psd_norm * np.log2(psd_norm + eps))
    return np.array([mean_val, std_val, rms_val, max_val, min_val, p2p_val,
                     skew_val, kurt_val, crest_val, shape_val, impulse_val,
                     freq_dom, spec_energy, spec_entropy], dtype=np.float32)

def file_to_features(fpath):
    """Charge un fichier CSV et extrait les features par fenêtres glissantes."""
    df     = pd.read_csv(fpath, header=0)
    signal = df.iloc[:, 0].values.astype(np.float32)  # canal horizontal
    feats  = []
    for start in range(0, len(signal) - WINDOW_SIZE, WINDOW_STEP):
        feats.append(extract_features(signal[start:start+WINDOW_SIZE]))
    return np.mean(feats, axis=0) if feats else None

print("Fonctions prêtes ✓")

In [ ]:
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow import keras
import pandas as pd

LOG_FEATURES = ["freq_dominant", "spectral_energy"]

# ════════════════════════════════════════════════════════════════════════════
# PHASE 1 : RODAGE  (fichiers 1 → 30 ignorés)
# ════════════════════════════════════════════════════════════════════════════
print("═"*55)
print("  PHASE 1 — RODAGE (fichiers 1→30 ignorés)")
print("═"*55)

for i, f in enumerate(files[:WARMUP_END]):
    print(f"\r  ⏳ Rodage : {i+1}/{WARMUP_END} ({f.name})   ", end="", flush=True)

print(f"\n  ✓ Rodage terminé — {WARMUP_END} fichiers ignorés\n")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 2 : COLLECTE  (fichiers 31 → 120)
# ════════════════════════════════════════════════════════════════════════════
print("═"*55)
print(f"  PHASE 2 — COLLECTE (fichiers {WARMUP_END+1}→{COLLECT_END})")
print("═"*55)

collect_files = files[WARMUP_END:COLLECT_END]
features_list = []

for i, fpath in enumerate(collect_files):
    feat = file_to_features(fpath)
    if feat is not None:
        features_list.append(feat)
    print(f"\r  📡 Collecte : {i+1}/{len(collect_files)} fichiers"
          f" | {len(features_list)} vecteurs   ", end="", flush=True)

features_array = np.array(features_list, dtype=np.float32)
print(f"\n  ✓ Collecte terminée : {features_array.shape[0]} vecteurs\n")

# Vérification stats brutes
print("── Stats brutes (avant log) ──")
df_check = pd.DataFrame(features_array, columns=FEATURE_NAMES)
print(df_check.describe().T[['mean','std','min','max']].round(2))

# ── Transformation log sur features à grande échelle ─────────────────────
features_log = features_array.copy()
for feat_name in LOG_FEATURES:
    idx = FEATURE_NAMES.index(feat_name)
    features_log[:, idx] = np.log10(np.abs(features_array[:, idx]) + 1e-10)

print("\n── Stats après log ──")
df_log = pd.DataFrame(features_log, columns=FEATURE_NAMES)
print(df_log.describe().T[['mean','std','min','max']].round(3))

# ════════════════════════════════════════════════════════════════════════════
# ENTRAÎNEMENT
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("  ENTRAÎNEMENT — Autoencodeur sur données normales")
print("═"*55)

scaler  = RobustScaler()
X_train = scaler.fit_transform(features_log)

inp   = keras.Input(shape=(14,))
enc   = keras.layers.Dense(8, activation="relu")(inp)
enc   = keras.layers.Dense(4, activation="relu")(enc)
dec   = keras.layers.Dense(8, activation="relu")(enc)
out   = keras.layers.Dense(14, activation="linear")(dec)
model = keras.Model(inp, out)
model.compile(optimizer="adam", loss="mse")

history = model.fit(
    X_train, X_train,
    epochs=100, batch_size=32,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=0
)
print(f"  Epochs : {len(history.history['loss'])}"
      f" | Loss : {history.history['loss'][-1]:.5f}"
      f" | Val : {history.history['val_loss'][-1]:.5f}")

recon     = model.predict(X_train, verbose=0)
errors    = np.mean((X_train - recon)**2, axis=1)
mean_err  = float(np.mean(errors))
std_err   = float(np.std(errors))
threshold = mean_err + 3.0 * std_err

print(f"  Seuil : {threshold:.5f}  (mean={mean_err:.5f} + 3σ={3*std_err:.5f})")
print(f"  ✓ Modèle prêt\n")

# ════════════════════════════════════════════════════════════════════════════
# PHASE 3 : DÉTECTION  (fichiers 121 → 2538)
# ════════════════════════════════════════════════════════════════════════════
print("═"*55)
print(f"  PHASE 3 — DÉTECTION (fichiers {COLLECT_END+1}→{len(files)})")
print("═"*55)

detect_files  = files[COLLECT_END:]
detect_errors = []
detect_labels = []
detect_idx    = []
first_anomaly = None

for i, fpath in enumerate(detect_files):
    feat = file_to_features(fpath)
    if feat is None:
        continue

    # Même transformation log qu'à l'entraînement
    feat_log = feat.copy()
    for feat_name in LOG_FEATURES:
        idx = FEATURE_NAMES.index(feat_name)
        feat_log[idx] = np.log10(np.abs(feat[idx]) + 1e-10)

    feat_s  = scaler.transform(feat_log.reshape(1, -1))
    recon   = model.predict(feat_s, verbose=0)
    err     = float(np.mean((feat_s - recon)**2))
    anomaly = err > threshold

    detect_errors.append(err)
    detect_labels.append(anomaly)
    detect_idx.append(COLLECT_END + i + 1)

    if anomaly and first_anomaly is None:
        first_anomaly = COLLECT_END + i + 1
        print(f"\n  🔴 PREMIÈRE ANOMALIE au fichier #{first_anomaly} "
              f"(minute {first_anomaly}) | erreur={err:.4f}")

    status = "🔴 ANOMALIE" if anomaly else "🟢 NORMAL  "
    print(f"\r  {status} | fichier {fpath.name:<8} | "
          f"erreur={err:.4f} | seuil={threshold:.4f}   ",
          end="", flush=True)

n_anomalies = sum(detect_labels)
print(f"\n\n  ✓ Détection terminée")
print(f"  Fichiers analysés   : {len(detect_labels)}")
print(f"  Anomalies détectées : {n_anomalies} ({100*n_anomalies/max(len(detect_labels),1):.1f}%)")
if first_anomaly:
    print(f"  Première anomalie   : fichier #{first_anomaly} = minute {first_anomaly}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# ── Courbe d'entraînement ─────────────────────────────────────────────────
axes[0].plot(history.history['loss'],     label="Train", color="steelblue")
axes[0].plot(history.history['val_loss'], label="Val",   color="orange")
axes[0].set_title("Entraînement — données normales (fichiers 31→120)", fontsize=11)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# ── Erreurs pendant la collecte ───────────────────────────────────────────
axes[1].plot(errors, color="steelblue", alpha=0.7, linewidth=0.8)
axes[1].axhline(threshold, color="red",    linestyle="--", linewidth=1.5,
                label=f"Seuil = {threshold:.4f}")
axes[1].axhline(mean_err,  color="orange", linestyle="--", linewidth=1,
                label=f"Moyenne = {mean_err:.4f}")
axes[1].set_title("Erreurs de reconstruction — phase normale (collecte)", fontsize=11)
axes[1].set_ylabel("MSE")
axes[1].set_xlabel("Vecteur de features")
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

# ── Détection temps réel (échelle log) ───────────────────────────────────
colors_d = ["tomato" if a else "steelblue" for a in detect_labels]
axes[2].scatter(detect_idx, detect_errors, c=colors_d, s=10, alpha=0.7)
axes[2].axhline(threshold, color="red", linestyle="--", linewidth=1.5,
                label=f"Seuil = {threshold:.4f}")
if first_anomaly:
    axes[2].axvline(first_anomaly, color="black", linestyle=":", linewidth=1.5,
                    label=f"1ère anomalie : fichier #{first_anomaly}")
axes[2].set_yscale('log')
axes[2].set_title(
    f"Détection sur Bearing3_1 — {n_anomalies}/{len(detect_labels)} anomalies "
    f"(échelle log)", fontsize=11)
axes[2].set_xlabel("Numéro de fichier (= minute)")
axes[2].set_ylabel("Erreur MSE (log)")

from matplotlib.patches import Patch
axes[2].legend(handles=[
    Patch(color="steelblue", label="Normal"),
    Patch(color="tomato",    label="Anomalie"),
    plt.Line2D([0],[0], color="red",   linestyle="--", label=f"Seuil = {threshold:.4f}"),
    plt.Line2D([0],[0], color="black", linestyle=":",  label=f"1ère anomalie : #{first_anomaly}"),
])
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig("detection_bearing3_1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé → detection_bearing3_1.png ✓")

In [ ]:
import pandas as pd

# Vérifie les stats des features de collecte
df_feat = pd.DataFrame(features_array, columns=FEATURE_NAMES)
print(df_feat.describe().T[['mean','std','min','max']].round(3))